# Pueblo, CO — Land Value Return Analysis

Revenue-neutral **4:1 split-rate** (land taxed 4x improvements) on the **City of
Pueblo's own municipal mill levy only**.

## Policy scope (Greg Miller, 2026-08-20)

| Choice | Setting |
|---|---|
| Levy scope | **City of Pueblo's own 15.633-mill municipal levy ONLY.** Pueblo County (30.098), Pueblo City Schools D60 (41.402), Pueblo Library District (6.197), SE Colorado Water Conservancy (0.747), the Conservancy District and every special/urban-renewal district are held UNCHANGED. |
| Reform | Revenue-neutral split-rate, land:improvement = **4:1** |
| Exemptions | Preserve all. Fully-exempt parcels held out of the solver, the export and the charts. Partial (senior/veteran) exemptions preserved. |
| Roll year | **2025** (taxes payable 2026) |

## The city levy is a single general-operating levy

City of Pueblo Ordinance No. 11074 (adopted 2025-11-24) certifies one line —
`Gross Mill Levy 15.633 mills` — and the 2024 ACFR states the levy's *"sole
purpose is general operating expenses. No mill levy has been approved for debt
service or other purposes."* The rate has been 15.633 mills every year since at
least 2015. The city is only **16.6%** of the ~94.077-mill stack on a typical
Pueblo parcel, so this models a shift inside one sixth of the bill.

Two 5.000-mill special improvement maintenance districts (Southpointe, Bandera
Boulevard) sit inside the city but are **not** the citywide levy and are left
alone.

## What replaced the previous placeholder model

The notebook that previously sat here was an acknowledged placeholder: a
**uniform 1.0-mill proxy** current tax ("Absolute dollar values are not real"),
a 2:1 ratio, and classification off the `Zoning` attribute — which is **blank
for 100% of parcels** in the county feed, collapsing every parcel into one
"Other" bucket. All three are replaced here:

1. **Real current tax** — the certified 15.633-mill city levy applied to the
   assessor's own assessed values.
2. **4:1** ratio.
3. **Real classification** from three independent sources that agree with each
   other (Section 3).


## Section 1 — Imports and setup

In [ ]:
import sys
import json
import os
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from shapely.geometry import Polygon, MultiPolygon

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'pueblo'
STATE_FIPS = '08'
COUNTY_FIPS = '101'                  # Pueblo County
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0

# City of Pueblo's own certified municipal levy, roll year 2025 (payable 2026).
# Ord. No. 11074 (2025-11-24); confirmed against Colorado DOLA's DLG-70 compilation
# (PropTaxMap2025 layer 46, lgid 51016 "Pueblo, City of": GENERAL_LEVY = TOTAL_LEVY
# = 15.633, every other fund line zero).
CITY_MILLAGE = 15.633
ROLL_YEAR = 2025

# Colorado assessment rates, assessment year 2025, LOCAL-GOVERNMENT (non-school) basis.
# NOT assumed — reverse-engineered from the assessor's own assessed/actual pairs in
# Section 3, then confirmed against the Division of Property Taxation's 2025 rate sheet.
RES_RATE_LOCAL = 0.0625              # residential, flat, NO actual-value exclusion in 2025
NONRES_RATE = 0.2700                 # vacant land, commercial, industrial, agricultural

PARCEL_ID_COL = 'PAR_TXT'
OWNER_NAME_COL = 'Owner'
OWNER_ADDRESS_COL = 'SITEADDRESS'
PARCEL_URL_TEMPLATE = None           # county assessor portal has no stable per-parcel deep link

DATA_DIR = Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

ARCGIS = 'https://services1.arcgis.com/IL17xsvNU5Bmw3RY/ArcGIS/rest/services'
PARCELS_URL = f'{ARCGIS}/County_Parcels/FeatureServer/0/query'
ZONING_URL = f'{ARCGIS}/Zoning_Districts/FeatureServer/40/query'
ADDR_URL = f'{ARCGIS}/City_Address_Points_Dynamic/FeatureServer/30/query'
# Pueblo's own City_Limits_* services return null geometry or a 250 m fragment; use
# Census TIGER 2025 Incorporated Places. Pueblo city = place GEOID 0862000.
CITY_LIMITS_URL = ('https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/'
                   'Places_CouSub_ConCity_SubMCD/MapServer/4/query')
CITY_LIMITS_WHERE = "GEOID='0862000'"

print(f'{CITY_NAME} | roll {ROLL_YEAR} | city levy {CITY_MILLAGE} mills | ratio {LAND_IMPROVEMENT_RATIO}:1')
print(f"CENSUS_API_KEY loaded: {bool(os.getenv('CENSUS_API_KEY'))}")

## Section 2 — Fetch parcels, zoning, address points, building footprints

Four sources, all open, all cached in `data/`:

| Source | What it gives |
|---|---|
| Pueblo County `County_Parcels/0` (101,040 parcels) | owner, `TaxDistrict`, `Neighborhood`, `TaxExempt`, `SeniorExemption`, land + improvement **assessed and actual** value, geometry |
| City of Pueblo `Zoning_Districts/40` (1,253 polygons) | the zoning district actually in force — the county's own parcel-level `Zoning` attribute is blank for 100% of parcels |
| City of Pueblo `City_Address_Points_Dynamic/30` (90,706 points) | `ASSESSOR_PARTEX` joins straight to the parcel; `APT` gives **dwelling-unit counts**, `FACILITYSUBTYPE` a land-use class, `FULLADDR` the situs address |
| Overture Maps buildings, release 2026-08-19.0 | building **footprint count and area** per parcel |

**Levy scope comes from `TaxDistrict`, not from a polygon.** The assessor stamps
every parcel with the tax district that determines which levies it pays; codes
beginning `60B` (School District 60 + City of Pueblo) or `70V` (District 70 +
City of Pueblo) are exactly the parcels that pay the City of Pueblo's mills.
That is the levy scope by definition, so it is used as the city filter and the
TIGER place boundary is kept only as a cross-check.

In [ ]:
def esri_to_shapely(geom):
    """ESRI ring array -> shapely Polygon/MultiPolygon (clockwise = outer, ccw = hole)."""
    rings = (geom or {}).get('rings')
    if not rings:
        return None
    outer, holes = [], []
    for r in rings:
        if len(r) < 4:
            continue
        area = 0.0
        for i in range(len(r) - 1):
            area += r[i][0] * r[i + 1][1] - r[i + 1][0] * r[i][1]
        (holes if area > 0 else outer).append(r)
    if not outer:
        outer, holes = [r for r in rings if len(r) >= 4], []
    if not outer:
        return None
    try:
        polys = [Polygon(o, holes if len(outer) == 1 else None) for o in outer]
    except Exception:
        return None
    polys = [p for p in polys if not p.is_empty]
    if not polys:
        return None
    return polys[0] if len(polys) == 1 else MultiPolygon(polys)


def fetch_arcgis(url, out_fields, where='1=1', page=2000, geometry=True, label=''):
    """Paged ArcGIS FeatureServer query with retries."""
    session = requests.Session()
    total = int(session.get(url, params={'f': 'json', 'where': where,
                                        'returnCountOnly': 'true'}, timeout=90).json().get('count', 0))
    print(f'{label}: {total:,} records', flush=True)
    rows, offset, t0 = [], 0, time.time()
    while offset < total:
        params = {'f': 'json', 'where': where, 'outFields': ','.join(out_fields),
                  'returnGeometry': str(geometry).lower(), 'resultOffset': offset,
                  'resultRecordCount': page, 'orderByFields': 'OBJECTID ASC'}
        if geometry:
            params.update({'outSR': 4326, 'geometryPrecision': 7})
        for attempt in range(4):
            try:
                resp = session.get(url, params=params, timeout=240)
                resp.raise_for_status()
                feats = resp.json().get('features', [])
                break
            except Exception as exc:
                print(f'  retry {attempt} @{offset}: {exc}', flush=True)
                time.sleep(4 * (attempt + 1))
        else:
            raise RuntimeError(f'{label} failed at offset {offset}')
        if not feats:
            break
        for f in feats:
            attrs = dict(f.get('attributes', {}))
            if geometry:
                g = esri_to_shapely(f.get('geometry') or {})
                if g is None:
                    continue
                attrs['geometry'] = g
            rows.append(attrs)
        offset += page
    print(f'{label}: kept {len(rows):,} in {time.time() - t0:.0f}s', flush=True)
    return rows

In [ ]:
PARCEL_FIELDS = [
    'OBJECTID', 'PAR_TXT', 'PAR_NUM', 'Owner', 'OwnerStreetAddress', 'OwnerCity',
    'OwnerState', 'OwnerZip', 'TaxDistrict', 'Neighborhood', 'Subdivision',
    'TaxExempt', 'MobileHomePresent', 'SeniorExemption', 'Zoning',
    'LegalDescription', 'LandAssessedValue', 'LandActualValue',
    'ImprovementsAssessedValue', 'ImprovementsActualValue', 'Tax',
    'AssessorURL', 'Shape__Area',
]

RAW_PARCELS = DATA_DIR / 'parcels_raw.gpq'
if RAW_PARCELS.exists():
    county = gpd.read_parquet(RAW_PARCELS)
    print(f'Loaded {len(county):,} county parcels from cache')
else:
    county = gpd.GeoDataFrame(
        fetch_arcgis(PARCELS_URL, PARCEL_FIELDS, label='county parcels'),
        geometry='geometry', crs='EPSG:4326')
    county.to_parquet(RAW_PARCELS, index=False)
    print(f'Fetched and cached {len(county):,} county parcels')

ZONING_PATH = DATA_DIR / 'zoning_districts.parquet'
if ZONING_PATH.exists():
    zoning = gpd.read_parquet(ZONING_PATH)
else:
    zoning = gpd.GeoDataFrame(
        fetch_arcgis(ZONING_URL, ['OBJECTID', 'ZONING', 'ORD'], label='zoning districts'),
        geometry='geometry', crs='EPSG:4326')
    zoning.to_parquet(ZONING_PATH, index=False)
zoning['ZONING'] = zoning['ZONING'].astype(str).str.strip()
print(f'{len(zoning):,} zoning polygons, {zoning["ZONING"].nunique()} districts')

ADDR_PATH = DATA_DIR / 'address_points.parquet'
if ADDR_PATH.exists():
    addr = pd.read_parquet(ADDR_PATH)
else:
    addr = pd.DataFrame(fetch_arcgis(
        ADDR_URL,
        ['OBJECTID', 'FULLADDR', 'ADDRNUM', 'APT', 'TYPE', 'FACILITYNAME',
         'FACILITYSUBTYPE', 'ASSESSOR_PARID', 'ASSESSOR_PARTEX', 'Status', 'NEIGH'],
        geometry=False, label='address points'))
    addr.to_parquet(ADDR_PATH, index=False)
print(f'{len(addr):,} address points')

CITY_LIMITS_PATH = DATA_DIR / 'pueblo_city_limits.parquet'
if CITY_LIMITS_PATH.exists():
    city_limits = gpd.read_parquet(CITY_LIMITS_PATH)
else:
    city_limits = gpd.GeoDataFrame(
        fetch_arcgis(CITY_LIMITS_URL, ['GEOID', 'NAME'], where=CITY_LIMITS_WHERE,
                     label='TIGER place'),
        geometry='geometry', crs='EPSG:4326')
    city_limits.to_parquet(CITY_LIMITS_PATH, index=False)

In [ ]:
# Overture Maps building footprints for the Pueblo bbox. Bounded temp/memory so a
# spill cannot fill the disk. Skipped gracefully if duckdb or the network is absent —
# footprints only refine the residential unit-count fallback, they are not load-bearing.
BUILDINGS_PATH = DATA_DIR / 'buildings.parquet'
if not BUILDINGS_PATH.exists():
    try:
        import duckdb
        con = duckdb.connect()
        con.execute("INSTALL spatial; LOAD spatial; INSTALL httpfs; LOAD httpfs; "
                    "SET s3_region='us-west-2';")
        con.execute("SET max_temp_directory_size='4GiB'; SET memory_limit='6GB'; "
                    "SET preserve_insertion_order=false;")
        con.execute(f"""
            COPY (
              SELECT id, height, num_floors, class, subtype, ST_AsWKB(geometry) AS geom_wkb
              FROM read_parquet('s3://overturemaps-us-west-2/release/2026-08-19.0/'
                                'theme=buildings/type=building/*.parquet')
              WHERE bbox.xmin > -104.73 AND bbox.xmax < -104.45
                AND bbox.ymin > 38.15  AND bbox.ymax < 38.35
            ) TO '{BUILDINGS_PATH}' (FORMAT PARQUET);
        """)
        print('Fetched Overture building footprints')
    except Exception as exc:
        print(f'Building footprints unavailable ({exc}) — unit-count fallback will use '
              f'improvement value only')

if BUILDINGS_PATH.exists():
    _b = pd.read_parquet(BUILDINGS_PATH)
    buildings = gpd.GeoDataFrame(
        _b.drop(columns=['geom_wkb']),
        geometry=gpd.GeoSeries.from_wkb(_b['geom_wkb']), crs='EPSG:4326')
    print(f'{len(buildings):,} building footprints')
else:
    buildings = None

In [ ]:
VALUE_COLS = ['LandAssessedValue', 'LandActualValue',
              'ImprovementsAssessedValue', 'ImprovementsActualValue']
for col in VALUE_COLS:
    county[col] = pd.to_numeric(county[col], errors='coerce').fillna(0)

county['tax_district'] = county['TaxDistrict'].astype(str).str.strip()

# Levy scope: parcels that pay the City of Pueblo's mills.
CITY_TD = r'^(60B|70V)'
county['in_city_levy'] = county['tax_district'].str.match(CITY_TD).fillna(False)

# Cross-check against the TIGER place boundary (centroid-in-polygon).
_city_3857 = city_limits.to_crs(3857)[['geometry']]
_p3857 = county.to_crs(3857)
_pts = gpd.GeoDataFrame(_p3857[['OBJECTID']].copy(),
                        geometry=_p3857.geometry.representative_point(), crs=3857)
_hit = gpd.sjoin(_pts, _city_3857, how='inner', predicate='within')
county['in_tiger_place'] = county['OBJECTID'].isin(set(_hit['OBJECTID'].astype(int)))

print('TaxDistrict vs TIGER place boundary:')
print(pd.crosstab(county['in_city_levy'], county['in_tiger_place']))
_a = county.loc[county['in_city_levy'] & ~county['in_tiger_place'],
                ['LandAssessedValue', 'ImprovementsAssessedValue']].sum().sum()
_b = county.loc[~county['in_city_levy'] & county['in_tiger_place'],
                ['LandAssessedValue', 'ImprovementsAssessedValue']].sum().sum()
print(f'\nCity levy district but outside TIGER place: ${_a:,.0f} assessed')
print(f'Inside TIGER place but not a city levy district: ${_b:,.0f} assessed '
      f'(unincorporated enclaves — correctly excluded, they pay no city mills)')

gdf = county[county['in_city_levy']].copy().reset_index(drop=True)
print(f'\nCity of Pueblo parcels (levy basis): {len(gdf):,} of {len(county):,} county parcels')

## Section 3 — Derive the assessment formula, then classify

### 3a. Reverse-engineer Colorado's assessment rates from the assessor's own numbers

Colorado taxes **assessed** value, and the assessment rate differs by class *and*
by levy type (school vs. local government). The rates are not assumed here: they
are derived from the ratio of the assessor's `*AssessedValue` to its
`*ActualValue`, per parcel, and only then checked against the statute.

In [ ]:
gdf['land_actual'] = gdf['LandActualValue'].clip(lower=0)
gdf['impr_actual'] = gdf['ImprovementsActualValue'].clip(lower=0)
gdf['land_assessed'] = gdf['LandAssessedValue'].clip(lower=0)
gdf['impr_assessed'] = gdf['ImprovementsAssessedValue'].clip(lower=0)
gdf['total_actual'] = gdf['land_actual'] + gdf['impr_actual']
gdf['total_assessed'] = gdf['land_assessed'] + gdf['impr_assessed']

_m = gdf['total_actual'] > 0
_ratio = pd.Series(np.nan, index=gdf.index)
_ratio[_m] = gdf.loc[_m, 'total_assessed'] / gdf.loc[_m, 'total_actual']
gdf['assessment_ratio'] = _ratio

print('Observed assessed/actual ratio — most common values:')
print(_ratio.round(4).value_counts().head(8).to_string())

# Does the ratio vary with value? A 2026-style "lesser of 10% or $70,000" exclusion
# would make the effective rate rise with value. A flat rate rules the exclusion out.
_res = gdf[np.isclose(_ratio, RES_RATE_LOCAL, atol=0.0015)].copy()
_res['band'] = pd.cut(_res['total_actual'],
                      [0, 1e5, 3e5, 7e5, 1e6, 5e6, 1e13],
                      labels=['<100k', '100-300k', '300-700k', '700k-1M', '1-5M', '>5M'])
print('\nResidential-rate parcels, mean ratio by value band '
      '(flat => no actual-value exclusion):')
print(_res.groupby('band', observed=True)['assessment_ratio'].agg(['size', 'mean']).round(6).to_string())

_res_hit = np.isclose(_ratio, RES_RATE_LOCAL, atol=0.0015).sum()
_non_hit = np.isclose(_ratio, NONRES_RATE, atol=0.0025).sum()
_valued = int(_m.sum())
print(f'\nParcels reproduced EXACTLY by one of the two statutory rates: '
      f'{_res_hit + _non_hit:,} of {_valued:,} valued parcels '
      f'({100 * (_res_hit + _non_hit) / _valued:.2f}%)')
print(f'  residential {RES_RATE_LOCAL:.4%}: {_res_hit:,}   '
      f'non-residential {NONRES_RATE:.2%}: {_non_hit:,}')
print(f'  remainder {_valued - _res_hit - _non_hit:,} sit BETWEEN the two rates = '
      f'mixed-use parcels (C.R.S. 39-1-103(9)(a) values the residential and '
      f'non-residential portions separately, so the blended ratio lands in between)')

**Result — the parcel file is roll year 2025 (payable 2026), local-government basis.**

Residential is a flat **6.25%** of actual value across every value band from
under \$100k to over \$5M, and vacant/commercial/industrial is a flat **27.00%**.
That combination is unique to assessment year 2025, and it independently dates
the file:

| Assessment year | Residential (local gov) | Actual-value exclusion | Vacant | Commercial |
|---|---|---|---|---|
| 2023 | 6.7% | −$55,000 | 27.9% | 27.9% |
| 2024 | 6.7% | −$55,000 | 27.9% | 27.9% |
| **2025** | **6.25%** | **none** | **27.0%** | **27.0%** |
| 2026 | 6.8% | −10% of first \$700k (max \$70,000) | 26% | 25% |

Source: Colorado Division of Property Taxation 2025 rate sheet and the DPT 2025
(Fifty-Fifth) Annual Report p. 30 — HB24B-1001 made 2025's residential rates
contingent on statewide actual-value growth; the State Board certified growth of
4.27%, so the "≤5%" branch governs (school 7.05%, local government 6.25%).
Because we are modelling the **city** (non-school) levy, the local-government
basis is the correct one, and it is what the assessor has published here.

The 2025 roll year matches the roll year the 15.633-mill levy was struck on
(Ord. 11074), so **no derived-rate adjustment is needed** — the published rate
is applied directly.

### 3b. Classify — three independent sources that agree

The county's parcel feed has **no property-class or land-use code field**: its
`Zoning`, `Tax` and `LevyURL` attributes come back blank or zero for 100% of
parcels, and `AssessorURL` is empty. Classification is therefore built from
three genuinely independent, non-heuristic sources, cross-tabulated below to
show they agree:

1. **The assessor's own statutory class**, recovered from which assessment rate
   it applied (6.25% = residential, 27% = non-residential, in between =
   mixed-use). This is the assessor's classification decision, just encoded
   arithmetically rather than in a code column.
2. **The zoning district in force**, from the city's `Zoning_Districts` polygon
   layer (100% of parcels match). District meanings are from Pueblo Municipal
   Code Title XVII § 17-4-51 and the § 17-4-51(c) Table of Uses.
3. **The city's address-point land-use class** (`FACILITYSUBTYPE`) and, for the
   residential split, the **count of distinct addresses and unit designators**
   on the parcel — an apartment building carries one address point per unit.

Dwelling-unit counts are what separate single-family from multi-family, and they
come from real addressing data, not from a value threshold.

In [ ]:
# --- zoning (representative point in polygon) ---
_z3857 = zoning.to_crs(3857)[['ZONING', 'geometry']]
_g3857 = gdf.to_crs(3857)
_pts = gpd.GeoDataFrame(_g3857[['OBJECTID']].copy(),
                        geometry=_g3857.geometry.representative_point(), crs=3857)
_zj = gpd.sjoin(_pts, _z3857, how='left', predicate='within').drop_duplicates('OBJECTID')
gdf = gdf.merge(_zj[['OBJECTID', 'ZONING']], on='OBJECTID', how='left')
gdf['ZONING'] = gdf['ZONING'].fillna('')
print(f'Zoning matched: {100 * (gdf["ZONING"] != "").mean():.1f}% of city parcels')

# --- land area (projected) ---
gdf['lot_sqft'] = gdf.to_crs(3857).geometry.area * 10.763910417
gdf['lot_acres'] = gdf['lot_sqft'] / 43560.0

# --- address points: unit counts, situs address, facility class ---
_a = addr[addr['Status'].astype(str).str.strip().str.upper() == 'ACTIVE'].copy()
_a['par'] = _a['ASSESSOR_PARTEX'].astype(str).str.strip()
_a['apt'] = _a['APT'].astype(str).str.strip().replace({'None': '', 'nan': ''})
_a['has_unit'] = _a['apt'].str.len() > 0
_a['full'] = _a['FULLADDR'].astype(str).str.strip()
# A mobile-home / RV park addresses its pads "SPACE nnn" or "LOT nnn".
_a['is_pad'] = _a['full'].str.upper().str.contains(r'\bSPACE\b|\bTRLR\b', regex=True)
_agg = _a.groupby('par').agg(
    addr_count=('OBJECTID', 'size'),
    unit_count=('has_unit', 'sum'),
    pad_count=('is_pad', 'sum'),
    site_address=('full', 'first'),
    facility_subtype=('FACILITYSUBTYPE', 'median'),
    facility_name=('FACILITYNAME', 'first'),
).reset_index()

gdf['par'] = gdf['PAR_TXT'].astype(str).str.strip()
gdf = gdf.merge(_agg, left_on='par', right_on='par', how='left')
gdf['addr_count'] = gdf['addr_count'].fillna(0).astype(int)
gdf['unit_count'] = gdf['unit_count'].fillna(0).astype(int)
gdf['pad_count'] = gdf['pad_count'].fillna(0).astype(int)
gdf['SITEADDRESS'] = gdf['site_address'].fillna('')
print(f'Parcels with >=1 active address point: '
      f'{100 * (gdf["addr_count"] > 0).mean():.1f}%')

# --- building footprints per parcel ---
gdf['bldg_count'] = 0
gdf['bldg_sqft'] = 0.0
if buildings is not None:
    _b3857 = buildings.to_crs(3857)
    _b3857['bsqft'] = _b3857.geometry.area * 10.763910417
    _bpts = gpd.GeoDataFrame(_b3857[['bsqft']].copy(),
                             geometry=_b3857.geometry.representative_point(), crs=3857)
    _parcels_only = gpd.GeoDataFrame(_g3857[['OBJECTID']].copy(),
                                     geometry=_g3857.geometry, crs=3857)
    _bj = gpd.sjoin(_bpts, _parcels_only, how='inner', predicate='within')
    _bagg = _bj.groupby('OBJECTID').agg(bldg_count=('bsqft', 'size'),
                                        bldg_sqft=('bsqft', 'sum')).reset_index()
    gdf = gdf.drop(columns=['bldg_count', 'bldg_sqft']).merge(_bagg, on='OBJECTID', how='left')
    gdf['bldg_count'] = gdf['bldg_count'].fillna(0).astype(int)
    gdf['bldg_sqft'] = gdf['bldg_sqft'].fillna(0.0)
    print(f'Parcels with >=1 building footprint: '
          f'{100 * (gdf["bldg_count"] > 0).mean():.1f}%  '
          f'(median footprint on built parcels: '
          f'{gdf.loc[gdf["bldg_count"] > 0, "bldg_sqft"].median():,.0f} sqft)')

In [ ]:
# --- exemptions ---
# TaxExempt = 'yes' parcels carry real actual values but $0 assessed value in the
# assessor's file, so the flag is internally consistent and can be trusted.
gdf['full_exmp'] = (gdf['TaxExempt'].astype(str).str.strip().str.lower() == 'yes').astype(int)
gdf['senior_exmp'] = (gdf['SeniorExemption'].astype(str).str.strip().str.lower() == 'yes').astype(int)

_ex = gdf[gdf['full_exmp'] == 1]
print(f'Fully exempt: {len(_ex):,} parcels — '
      f'${_ex["total_actual"].sum() / 1e6:,.1f}M actual value, '
      f'${_ex["total_assessed"].sum():,.0f} assessed '
      f'({int((_ex["total_assessed"] > 0).sum())} with any assessed value)')
print(f'Senior/veteran homestead exemption flagged: {gdf["senior_exmp"].sum():,} parcels')

In [ ]:
# --- statutory class from the applied assessment rate ---
def statutory_class(row):
    if row['impr_actual'] <= 0 and row['land_actual'] <= 0:
        return 'no_value'
    r = row['assessment_ratio']
    if pd.isna(r):
        return 'no_value'
    if np.isclose(r, RES_RATE_LOCAL, atol=0.0015):
        return 'residential'
    if np.isclose(r, NONRES_RATE, atol=0.0025):
        return 'nonresidential'
    if r < RES_RATE_LOCAL - 0.0015:
        return 'residential'        # below 6.25% = residential with a partial exemption
    return 'mixed'                   # blended 6.25% / 27% => mixed use

gdf['stat_class'] = gdf.apply(statutory_class, axis=1)

print('--- statutory class x zoning family (non-exempt, improved) ---')
ZFAM = {
    **{z: 'R single-fam only' for z in ['R1', 'R2', 'R2U', 'R8', 'A3', 'A4']},
    **{z: 'R duplex' for z in ['R3', 'RCN']},
    **{z: 'R multifamily' for z in ['R4', 'R5', 'R6']},
    'R7': 'R mobile-home park',
    **{z: 'Business' for z in ['B1', 'B2', 'B3', 'B4', 'CCN', 'HB',
                               'HARP1', 'HARP2', 'HARP3']},
    'O1': 'Office',
    **{z: 'Industrial' for z in ['I1', 'I2', 'I3', 'BP']},
    **{z: 'Agricultural' for z in ['A1', 'A2']},
    'S1': 'Governmental', 'S3': 'Flood plain', 'S4': 'Parking', 'S5': 'Open space',
    'PUD': 'PUD',
}
gdf['zone_family'] = gdf['ZONING'].map(ZFAM).fillna('Unzoned/other')
_built = gdf[(gdf['full_exmp'] == 0) & (gdf['impr_actual'] > 0)]
print(pd.crosstab(_built['zone_family'], _built['stat_class']).to_string())

print('\n--- statutory class x address-point facility class ---')
print(pd.crosstab(_built['facility_subtype'], _built['stat_class']).to_string())

The three sources line up: address-point `FACILITYSUBTYPE` **100** is
residential, **200** commercial, **300** industrial, and each lands almost
entirely in the matching zoning family and the matching statutory rate class.
Where they disagree the *statutory class* wins for the residential /
non-residential split (it is the assessor's own decision and it determines the
tax), and zoning plus facility class decide the sub-type.

### 3c. Two things a Colorado parcel file gets wrong on its face

Gate 5 caught both of these; each would have put a false claim on a slide.

**Surface parking is invisible in the assessor's data.** There is no parking use
code, and Pueblo's dedicated `S-4` Parking District covers only 7 polygons. A
surface lot therefore shows up as ordinary commercial land — or, if it carries no
improvement value, as "Vacant Land". OSM `amenity=parking` polygons are the only
open source that captures surface lots, so they are fetched and a parcel is
called parking when a lot covers **≥50% of it and buildings cover <10%**.

**Occupied mobile home parks look like vacant land.** Colorado titles
manufactured homes to their residents and taxes them as *personal property*, so
the park owner's parcel carries land value and **\$0 improvement value**. Eight
parks in Pueblo — one of them 284 rented pads on \$7.3M of land — were landing in
"Vacant Land" and posting the vacant-land increase. They are not vacant: people
live there. They are broken out as their own category, because burying them in
"Vacant Land" would have overstated the vacant-land story *and* hidden the fact
that a land-value shift lands hardest on the least wealthy housing in the city.

Breaking them out names the problem but does not fix it: a category is still
land-only if the parcel carries no building value. **Section 3d repairs the
underlying data** by joining the assessor's own per-home `MH` accounts back onto
the parcel each home stands on, so an occupied park is modeled as improved land
rather than as an empty lot.

In [ ]:
from lvt.transit_utils import fetch_osm_parking

# --- OSM surface parking ---
gdf['parking_coverage'] = 0.0
try:
    _park = fetch_osm_parking(city_limits, str(DATA_DIR / 'osm_parking.gpkg'),
                              to_crs='EPSG:3857')
    _park_union = _park.geometry.buffer(0).union_all()
    _g = gdf.to_crs(3857)
    _geom = _g.geometry.buffer(0)
    gdf['parking_coverage'] = (_geom.intersection(_park_union).area
                               / _geom.area.replace(0, np.nan)).fillna(0.0).values
    print(f'{len(_park):,} OSM parking polygons, {_park.area.sum() / 4046.86:,.0f} acres '
          f'inside the city')
except Exception as exc:
    print(f'OSM parking unavailable ({exc}) — parking category limited to zoning S-4')

gdf['bldg_coverage'] = (gdf['bldg_sqft'] / gdf['lot_sqft'].replace(0, np.nan)).fillna(0.0)
gdf['is_parking_lot'] = ((gdf['parking_coverage'] >= 0.50)
                         & (gdf['bldg_coverage'] < 0.10)) | (gdf['ZONING'] == 'S4')
print(f'Parcels identified as parking lots: {int(gdf["is_parking_lot"].sum()):,} '
      f'({int((gdf["ZONING"] == "S4").sum())} from the S-4 Parking District, '
      f'the rest from OSM coverage)')

# --- mobile home / manufactured housing parks ---
# R-7 is Pueblo's Mobile Home Residential district (PMC 17-4-51(10)); pad-style
# addressing ("SPACE nnn") identifies parks in other districts too.
gdf['is_mobile_home_park'] = (
    (gdf['pad_count'] >= 5)
    | ((gdf['ZONING'] == 'R7') & (gdf['addr_count'] >= 5))
)
_mh = gdf[gdf['is_mobile_home_park']]
print(f'\nMobile home / manufactured housing parks: {len(_mh):,} parcels, '
      f'{int(_mh["addr_count"].sum()):,} pads, '
      f'${_mh["land_actual"].sum() / 1e6:,.1f}M land value, '
      f'${_mh["impr_actual"].sum() / 1e6:,.1f}M improvement value')
print(f'  of which carry $0 improvement value (homes taxed as personal property): '
      f'{int((_mh["impr_actual"] <= 0).sum())}')

### The assessor's own records export — where the property class comes from

The county's ArcGIS parcel feed carries **no** property-class field (`Zoning`,
`Tax`, `LevyURL` and `AssessorURL` are blank or zero for 100% of parcels). But
the Pueblo County Assessor's **public property-search application** exports the
full roll, and that export carries the real thing:

| Field | Use here |
|---|---|
| `Land Use Code` | the **Colorado 4-digit abstract code** — the assessor's statutory classification |
| `Use Code` | account type (RE-RES / RE-COMM / RE-CONDO / RE-TH / RE-MU / RE-LAND / MH / MN / PI / SA) |
| `Primary Building Area`, `Building Count`, `Year Built` | real **building area**, not a footprint estimate |
| `Tax Year (h)` | **2025** on 107,527 of 108,127 records — the roll year, stated outright |
| `Building Value (h)`, `Total Land Value (h)` | actual values, which cross-check the GIS feed |
| `Exemption Codes` | named exemptions (`SENIOR`, `COMISCR`, …) |
| `Neighborhood Group` | a plain-language use grouping that corroborates the abstract code |

The export agrees with the GIS feed almost exactly — **98.5%** of parcels match
on land actual value and **99.7%** on improvement actual value — so the two are
describing the same roll. Geometry and the certified assessed values come from
the GIS feed; class, building area and year built come from the export.

Account types `MH` (mobile homes), `MN` (severed minerals), `PI` (possessory
interests) and `SA` (state-assessed utilities) have **no parcel geometry** —
6,783 such records exist. That is the direct, visible confirmation of why a
parcel file cannot reproduce the full certified roll.

In [ ]:
CAMA_PATH = DATA_DIR / 'assessor_cama.parquet'
CAMA_BASE = 'https://puebloco-search.gsacorp.io'
CAMA_USE_CODES = ['RE-RES', 'RE-COMM', 'RE-CONDO', 'RE-TH', 'RE-MU', 'RE-LAND',
                  'MH', 'MN', 'PI', 'SA']

if not CAMA_PATH.exists():
    import io
    import re as _re
    _frames = []
    for _code in CAMA_USE_CODES:
        for _attempt in range(3):
            try:
                _S = requests.Session()
                _S.headers.update({'User-Agent': 'LVTShift/1.0 (research)'})
                _tok = _re.findall(r'name="token"[^>]*value="([^"]+)"',
                                   _S.get(f'{CAMA_BASE}/search/adv', timeout=90).text)[0]
                _S.post(f'{CAMA_BASE}/search/adv',
                        data={'token': _tok, 'type': 'r', 'query[parcel][use][]': _code},
                        timeout=300)
                _e = _S.get(f'{CAMA_BASE}/export/adv/r',
                            params={'format': 'csv', 'sort': 'Property ID'}, timeout=600)
                _e.raise_for_status()
                _frames.append(pd.read_csv(
                    io.StringIO(_e.content.decode('utf-8-sig', errors='replace')),
                    dtype=str, low_memory=False))
                print(f'  {_code}: {len(_frames[-1]):,} rows', flush=True)
                break
            except Exception as _exc:
                print(f'  {_code} attempt {_attempt} failed: {_exc}', flush=True)
                time.sleep(10 * (_attempt + 1))
    pd.concat(_frames, ignore_index=True).to_parquet(CAMA_PATH, index=False)

cama = pd.read_parquet(CAMA_PATH)
cama['pid'] = cama['Property ID'].astype(str).str.replace('-', '', regex=False).str.strip()
print(f'Assessor export: {len(cama):,} records')
print(f'  roll year (Tax Year (h)): '
      f'{cama["Tax Year (h)"].value_counts().head(3).to_dict()}')
print(f'  account types without parcel geometry (MH/MN/PI/SA): '
      f'{int(cama["Use Code"].isin(["MH", "MN", "PI", "SA"]).sum()):,}')

# Real-property records only, newest roll year per parcel.
REAL_ACCOUNTS = ['RE-RES', 'RE-COMM', 'RE-CONDO', 'RE-TH', 'RE-MU', 'RE-LAND']
cama_re = (cama[cama['Use Code'].isin(REAL_ACCOUNTS)]
           .sort_values('Tax Year (h)', ascending=False)
           .drop_duplicates('pid', keep='first'))

CAMA_COLS = ['pid', 'Land Use Code', 'Use Code', 'Use Code Description',
             'Primary Building Area', 'Primary Building Year Built', 'Building Count',
             'Neighborhood Group', 'Exemption Codes', 'Site Address',
             'Building Value (h)', 'Total Land Value (h)', 'Parcel Size',
             'Parcel Size Units']
gdf['pid'] = gdf['PAR_TXT'].astype(str).str.strip()
gdf = gdf.merge(cama_re[CAMA_COLS], on='pid', how='left')
gdf = gdf.rename(columns={
    'Land Use Code': 'abstract_code', 'Use Code': 'account_type',
    'Primary Building Area': 'bldg_area_sqft', 'Building Count': 'bldg_count_cama',
    'Primary Building Year Built': 'year_built', 'Neighborhood Group': 'nbhd_group',
    'Exemption Codes': 'exemption_codes', 'Site Address': 'cama_site_address'})
for _c in ['bldg_area_sqft', 'bldg_count_cama', 'year_built',
           'Building Value (h)', 'Total Land Value (h)']:
    gdf[_c] = pd.to_numeric(gdf[_c], errors='coerce').fillna(0)

print(f'\nAbstract code present on {100 * gdf["abstract_code"].notna().mean():.1f}% '
      f'of city parcels')
_both = gdf[(gdf['land_actual'] > 0) & (gdf['Total Land Value (h)'] > 0)]
print(f'Value cross-check vs the GIS feed — land actual matches exactly on '
      f'{100 * (_both["land_actual"] == _both["Total Land Value (h)"]).mean():.1f}%')
print(f'Real building area present on '
      f'{100 * (gdf["bldg_area_sqft"] > 0).mean():.1f}% of parcels '
      f'(median {gdf.loc[gdf["bldg_area_sqft"] > 0, "bldg_area_sqft"].median():,.0f} sqft)')

### Classification from the Colorado abstract code

`PROPERTY_CATEGORY` now comes from the assessor's own 4-digit abstract code,
present on **99.2%** of city parcels. Zoning, the statutory assessment class and
the address-point facility class are retained as *corroboration* and are carried
into the export, but they no longer drive the category.

Two Pueblo-specific findings drive named categories that the standard list does
not contain, and both would otherwise ship a false claim:

- **Abstract code 1140 is Pueblo's mobile-home-park code.** All eight of the
  city's large manufactured-home communities carry it — one of them 284 rented
  pads on \$7.3M of land. Colorado titles manufactured homes to their residents
  and taxes them as **personal property**, so the park owner's parcel shows
  land value and **\$0 improvement value**, which the usual
  "\$0 improvement ⇒ Vacant Land" rule would call vacant. It is not vacant:
  people live there. Broken out as **Mobile Home Park** (18 parcels).
- **Abstract code 1135 is individual manufactured housing** (94 parcels), \$0
  improvement value for the same reason. Broken out as **Manufactured Housing**.

Both categories are set here **before** Section 3d puts the homes back on the
parcels, so the classification does not depend on the attribution and the
attribution cannot reshuffle the categories.

Everything else follows the standard Colorado schedule; any code not in the map
is reported rather than silently bucketed.

In [ ]:
ABSTRACT_MAP = {
    # --- vacant (0xxx) ---
    '0100': 'Vacant Land', '0101': 'Vacant Land', '0102': 'Vacant Land',
    '0200': 'Vacant Land', '0201': 'Vacant Land', '0300': 'Vacant Land',
    '0301': 'Vacant Land', '0400': 'Vacant Land', '0510': 'Vacant Land',
    '0520': 'Vacant Land', '0530': 'Vacant Land', '0540': 'Vacant Land',
    '0550': 'Vacant Land', '0560': 'Vacant Land', '0600': 'Vacant Land',
    '0700': 'Vacant Land',
    # --- residential (11xx) ---
    '1112': 'Single Family Residential', '1113': 'Single Family Residential',
    '1114': 'Single Family Residential', '1116': 'Single Family Residential',
    '1119': 'Single Family Residential',
    '1115': 'Small Multi-Family (2-4 units)',
    '1120': 'Large Multi-Family (5+ units)',   # CO 1120 = 4-8 units
    '1125': 'Large Multi-Family (5+ units)',   # 9+ units
    '1127': 'Large Multi-Family (5+ units)',
    '1129': 'Large Multi-Family (5+ units)',
    '1130': 'Condominium', '1131': 'Condominium', '1132': 'Condominium',
    '1135': 'Manufactured Housing',
    '1140': 'Mobile Home Park',
    '1177': 'Other Residential',
    # --- commercial (2xxx) ---
    '2112': 'Retail / General Commercial', '2113': 'Retail / General Commercial',
    '2114': 'Retail / General Commercial',
    '2115': 'Hotel',
    '2120': 'Office / Commercial Condo', '2121': 'Office / Commercial Condo',
    '2125': 'Other Commercial', '2130': 'Other Commercial',
    '2135': 'Other Commercial', '2145': 'Other Commercial',
    '2140': 'Mixed Use',
    # --- industrial (3xxx) ---
    '3112': 'Industrial', '3115': 'Industrial', '3120': 'Industrial',
    '3125': 'Industrial', '3130': 'Industrial',
}


def _cat_from_code(code):
    if not isinstance(code, str) or not code:
        return None
    if code.startswith('4'):
        return 'Agricultural'
    if code.startswith('5'):
        return 'Other'
    if code.startswith('9'):
        return 'Exempt'
    return ABSTRACT_MAP.get(code)


gdf['abstract_code'] = gdf['abstract_code'].fillna('').astype(str).str.strip()
gdf['cat_from_code'] = gdf['abstract_code'].map(_cat_from_code)

_unmapped = (gdf.loc[gdf['cat_from_code'].isna() & (gdf['abstract_code'] != ''),
                     'abstract_code'].value_counts())
if len(_unmapped):
    print(f'Abstract codes with no mapping ({int(_unmapped.sum())} parcels) — '
          f'falling back to the statutory class:')
    print(_unmapped.head(15).to_string())
else:
    print('Every abstract code present is mapped.')


def categorize(row):
    if row['full_exmp'] == 1:
        return 'Exempt'
    # .map() turns an unmapped code into NaN (a float), not None — so test for str.
    code_cat = row['cat_from_code'] if isinstance(row['cat_from_code'], str) else None
    # These three keep their identity even at $0 improvement value, because a $0
    # improvement value does not mean the land is empty.
    if code_cat == 'Mobile Home Park' or row['is_mobile_home_park']:
        return 'Mobile Home Park'
    if code_cat == 'Manufactured Housing':
        return 'Manufactured Housing'
    if row['is_parking_lot']:
        return 'Transportation - Parking'
    if row['impr_actual'] <= 0:
        return 'Vacant Land'
    if code_cat is not None and code_cat not in ('Exempt', 'Vacant Land'):
        return code_cat
    # No usable abstract code: fall back on the assessor's statutory rate class.
    if row['stat_class'] == 'mixed':
        return 'Mixed Use'
    if row['stat_class'] == 'nonresidential':
        if row['ZONING'] in {'I1', 'I2', 'I3', 'BP'} or row['facility_subtype'] == 300:
            return 'Industrial'
        if row['ZONING'] in {'B1', 'B2', 'B3', 'B4', 'CCN', 'HB',
                             'HARP1', 'HARP2', 'HARP3'}:
            return 'Retail / General Commercial'
        return 'Other Commercial'
    return 'Single Family Residential'


gdf['PROPERTY_CATEGORY'] = gdf.apply(categorize, axis=1)
assert gdf['PROPERTY_CATEGORY'].map(lambda v: isinstance(v, str)).all(), \
    'PROPERTY_CATEGORY must be all strings'

# Dwelling-unit estimate for residential parcels: distinct addresses on the parcel
# (an apartment building carries one address point per unit). Exported as LIVUNIT.
_units = gdf[['unit_count', 'addr_count']].max(axis=1).clip(lower=1)
gdf['est_units'] = np.where(gdf['stat_class'] == 'residential', _units, np.nan)

_nx = gdf[gdf['full_exmp'] == 0]
print('\n' + _nx['PROPERTY_CATEGORY'].value_counts().to_string())
print(f'\nUncategorised: {int(gdf["PROPERTY_CATEGORY"].isna().sum())}')

print('\n--- corroboration: abstract-code category x statutory rate class ---')
print(pd.crosstab(_nx['PROPERTY_CATEGORY'], _nx['stat_class']).to_string())

### 3d. Manufactured homes — put the house back on the parcel it stands on

Colorado titles a manufactured home to the person living in it and taxes it as
**personal** property unless the owner files a certificate of permanent location.
The consequence for a parcel file is severe and invisible: the park owner's
parcel carries the land value and **\$0 improvement value** even when 284
families live on it. Read literally, an occupied mobile home community looks
exactly like an empty lot, gets filed under the land-only ceiling, and posts the
vacant-land increase.

The assessor's own export carries the missing half. Alongside the real-property
accounts it publishes **2,783 `MH` accounts — one per home** — each with:

| Field | What it gives |
|---|---|
| `Building Value (h)` | the assessor's actual value for **that home** |
| `Primary Building Area` | the home's floor area (median 960 sqft) |
| `Site Address Number/Direction/Street Name/Suffix` | the **park's situs address** |
| `Site Address Unit` | the **pad** (`LOT 37`, `SPACE 212`) |
| `Neighborhood` | the park by name (`302 - COUNTRYSIDE ESTATES`) |
| `Net Assessed Value (h)` + `Assessed Exemption Value (h)` | taxable and exempt assessed value |

So the homes *can* be joined to the parcel they stand on, from the assessor's
own figures — no imputation, no per-pad estimate. Three passes, most specific
first:

1. **situs address + pad number** → the city's address-point layer (which
   carries one point per pad, each tagged with `ASSESSOR_PARTEX`, the parcel) →
   the parcel. Matches 1,150 of 1,293 city homes.
2. **situs address only**, resolved to the parcel holding the most address
   points at that address. Another 137.
3. **assessor neighborhood code** → the parcel the rest of that named park's
   homes already matched to. The last 6 (all address-typo residue:
   `3015 LAKE AVAE`, `2222 E 8TH ST ST`).

**1,293 of 1,293 (100%) join**, and they land where they should: 1,171 on the 18
Mobile Home Park parcels, 93 on the 94 individual manufactured-housing lots, and
25 on scattered parcels of other classes. The join is corroborated by a source
that had no part in building it — the city's own pad address points, which count
1,318 pads on those 18 parcels against 1,171 occupied homes. Homes never exceed
pads on any parcel.

**The modeling choice, stated plainly.** The home's **gross** assessed value is
added to the parcel's improvement value, in the **current** system and the
reformed one alike — the same even-handed treatment the senior exemption gets.
That means the \$28,000 low-value exemption of **C.R.S. 39-3-126.5** (HB22-1223)
is *not* carried across: it exempts 912 of the 1,293 homes (\$10.9M of actual
value, \$684,610 of assessed) from the **personal-property** roll, and that
exemption is a feature of the personal-property system, not of the real-property
roll a split rate is struck on. A tax on improvements reaches a manufactured home
because it is a building. The cost of the choice is bounded and reported below:
it adds **\$32,580 (0.20%)** to modeled current city revenue.

The check that this is right, not merely convenient: the residential base
validation against the assessor's own 2025 abstract goes from **−0.33% to
−0.03%**. The abstract's Residential class was counting these homes all along;
the model was the only thing missing them.

In [ ]:
# Manufactured homes are titled to their occupants and taxed as personal
# property, so the park owner's parcel shows land value and $0 improvement
# value. The assessor's own MH accounts carry the homes; join them back to the
# parcel they stand on and add their value to the parcel's improvements.
import re as _re2

_MH_UNIT_RE = _re2.compile(r'\s+(SPACE|SPC|SP|LOT|UNIT|APT|STE|TRLR|#)\.?\s*[\w-]*$')


def _mh_clean_addr(value: str) -> str:
    """Normalise a situs address to 'NUMBER DIR STREET SUFFIX', pad token removed."""
    s = str(value).upper().strip()
    s = _re2.sub(r'[.,]', ' ', s)
    s = _re2.sub(r'\s+', ' ', s).strip()
    s = _re2.sub(r'^0+\s+(?=\d)', '', s)          # '0 6464 DILLON DR' -> '6464 DILLON DR'
    for _ in range(3):                            # strip trailing pad tokens
        _s2 = _MH_UNIT_RE.sub('', s)
        if _s2 == s:
            break
        s = _s2
    s = _re2.sub(r'\b(\w+)( \1\b)+', r'\1', s)    # collapse '... ST ST'
    return _re2.sub(r'\s+', ' ', s).strip()


def _mh_unit_num(value: str) -> str:
    _d = _re2.findall(r'(\d+)', str(value).upper())
    return _d[-1] if _d else ''


_mh = cama[cama['Use Code'] == 'MH'].copy()
_mh['td'] = _mh['Tax District Code'].astype(str).str.strip()
_mh = _mh[_mh['td'].str.match(CITY_TD).fillna(False)].copy()
for _c in ['Building Value (h)', 'Extra Features Value (h)', 'Net Assessed Value (h)',
           'Assessed Exemption Value (h)', 'Primary Building Area']:
    _mh[_c] = pd.to_numeric(_mh[_c], errors='coerce').fillna(0)

# Actual value of the home; gross assessed value = taxable + exempted.
_mh['home_actual'] = _mh['Building Value (h)'] + _mh['Extra Features Value (h)']
_mh['home_assessed'] = _mh['Net Assessed Value (h)'] + _mh['Assessed Exemption Value (h)']
_mh['home_exempt'] = _mh['Assessed Exemption Value (h)']

_MH_ADDR_PARTS = ['Site Address Number', 'Site Address Direction',
                  'Site Address Street Name', 'Site Address Street Suffix',
                  'Site Address Post Direction']
def _mh_compose(row) -> str:
    _out = []
    for _v in row:
        _s = str(_v).strip().upper()
        if _s and _s not in ('NAN', 'NONE', '<NA>'):
            _out.append(_s)
    return ' '.join(_out)


_mh['akey'] = [_mh_clean_addr(_mh_compose(_r))
               for _r in _mh[_MH_ADDR_PARTS].itertuples(index=False, name=None)]
_mh['unum'] = _mh['Site Address Unit'].map(_mh_unit_num)
_mh['nbhd'] = _mh['Neighborhood Code'].astype(str).str.strip()

# Address points: one per pad, each tagged with the parcel it belongs to.
_ap = addr.copy()
_ap['par'] = _ap['ASSESSOR_PARTEX'].astype(str).str.strip()
_ap = _ap[_ap['par'].isin(set(gdf['par']))].copy()
_ap['akey'] = _ap['FULLADDR'].map(_mh_clean_addr)
_ap['unum'] = _ap['APT'].map(_mh_unit_num)


def _modal(s):
    return s.value_counts().index[0]


_k_addr_unit = _ap[_ap['unum'] != ''].groupby(['akey', 'unum'])['par'].agg(_modal)
_k_addr = _ap.groupby('akey')['par'].agg(_modal)

# Pass 1 — situs address + pad number.
_mh['par_match'] = pd.Series(list(zip(_mh['akey'], _mh['unum'])),
                             index=_mh.index).map(_k_addr_unit)
_mh['match_how'] = np.where(_mh['par_match'].notna(), 'addr+pad', '')
# Pass 2 — situs address alone.
_p2 = _mh['akey'].map(_k_addr)
_fill = _mh['par_match'].isna() & _p2.notna()
_mh.loc[_fill, 'par_match'] = _p2[_fill]
_mh.loc[_fill, 'match_how'] = 'addr'
# Pass 3 — assessor neighborhood (the named park) -> where its other homes landed.
_nbhd_map = _mh.loc[_mh['par_match'].notna()].groupby('nbhd')['par_match'].agg(_modal)
_fill = _mh['par_match'].isna() & _mh['nbhd'].isin(_nbhd_map.index)
_mh.loc[_fill, 'par_match'] = _mh.loc[_fill, 'nbhd'].map(_nbhd_map)
_mh.loc[_fill, 'match_how'] = 'neighborhood'
_mh['match_how'] = _mh['match_how'].replace('', 'unmatched')

print(f'City manufactured-home accounts (MH, tax district {CITY_TD}): {len(_mh):,}')
print(_mh['match_how'].value_counts().to_string())
_n_ok = int(_mh['par_match'].notna().sum())
print(f'joined to a modeled parcel: {_n_ok:,} of {len(_mh):,} '
      f'({100 * _n_ok / len(_mh):.1f}%)')
print(f'unjoined home value: ${_mh.loc[_mh["par_match"].isna(), "home_actual"].sum():,.0f}')

# C.R.S. 39-3-126.5 (HB22-1223): a mobile/manufactured home with actual value
# <= $28,000 is exempt from the personal-property roll. Confirm the cutoff in
# the assessor's own numbers rather than assuming it.
_abstex = _mh['Exemption Codes'].astype(str).str.contains('ABSTEX')
print(f'\nlow-value exempt (ABSTEX): {int(_abstex.sum()):,} homes, '
      f'max actual ${_mh.loc[_abstex, "Building Value (h)"].max():,.0f}; '
      f'taxable: {int((~_abstex).sum()):,} homes, '
      f'min actual ${_mh.loc[~_abstex, "Building Value (h)"].min():,.0f}')

_mh_agg = _mh[_mh['par_match'].notna()].groupby('par_match').agg(
    mh_home_count=('Property ID', 'size'),
    mh_home_actual=('home_actual', 'sum'),
    mh_home_assessed=('home_assessed', 'sum'),
    mh_home_exempt_assessed=('home_exempt', 'sum'),
    mh_home_sqft=('Primary Building Area', 'sum'),
)
gdf = gdf.merge(_mh_agg, left_on='par', right_index=True, how='left')
_MH_COLS = ['mh_home_count', 'mh_home_actual', 'mh_home_assessed',
            'mh_home_exempt_assessed', 'mh_home_sqft']
for _c in _MH_COLS:
    gdf[_c] = gdf[_c].fillna(0)
gdf['mh_home_count'] = gdf['mh_home_count'].astype(int)

print(f'\nparcels receiving manufactured-home value: '
      f'{int((gdf["mh_home_count"] > 0).sum()):,}')
print(gdf[gdf['mh_home_count'] > 0]
      .groupby('PROPERTY_CATEGORY')
      .agg(parcels=('mh_home_count', 'size'), homes=('mh_home_count', 'sum'),
           home_actual=('mh_home_actual', 'sum'),
           home_assessed=('mh_home_assessed', 'sum'))
      .sort_values('home_assessed', ascending=False).to_string())

# Corroboration from a source that had no part in the join: the city's own pad
# address points. Homes must never exceed pads.
_pk = gdf[gdf['PROPERTY_CATEGORY'] == 'Mobile Home Park']
print(f'\npark cross-check — {int(_pk["mh_home_count"].sum()):,} occupied homes vs '
      f'{int(_pk["addr_count"].sum()):,} pad address points on the same '
      f'{len(_pk)} parcels; parcels where homes exceed pads: '
      f'{int((_pk["mh_home_count"] > _pk["addr_count"]).sum())}')

# The attribution. Gross assessed value goes into the improvement base on BOTH
# sides of the comparison (current and reformed), because a tax on improvements
# reaches a manufactured home: it is a building. The $28,000 personal-property
# exemption is a feature of the personal-property roll, not of the real-property
# roll a split rate is struck on; its size is reported above and below.
_impr_actual_before = float(gdf['impr_actual'].sum())
_impr_assessed_before = float(gdf['impr_assessed'].sum())
gdf['impr_actual'] = gdf['impr_actual'] + gdf['mh_home_actual']
gdf['impr_assessed'] = gdf['impr_assessed'] + gdf['mh_home_assessed']
gdf['total_actual'] = gdf['land_actual'] + gdf['impr_actual']
gdf['total_assessed'] = gdf['land_assessed'] + gdf['impr_assessed']

# Building area only where the parcel had NO physical measure at all. Overture
# footprints already see most of these homes; overwriting a footprint with the
# assessor's floor area would make the coverage measure worse, not better.
_no_area = (gdf['bldg_area_sqft'] <= 0) & (gdf['bldg_sqft'] <= 0) & (gdf['mh_home_sqft'] > 0)
gdf.loc[_no_area, 'bldg_area_sqft'] = gdf.loc[_no_area, 'mh_home_sqft']
print(f'\nbuilding area filled from MH floor area on {int(_no_area.sum())} parcels '
      f'that had no assessor area and no Overture footprint')

print(f'\nimprovement actual value   ${_impr_actual_before:,.0f} '
      f'-> ${gdf["impr_actual"].sum():,.0f} '
      f'(+${gdf["mh_home_actual"].sum():,.0f})')
print(f'improvement assessed value ${_impr_assessed_before:,.0f} '
      f'-> ${gdf["impr_assessed"].sum():,.0f} '
      f'(+${gdf["mh_home_assessed"].sum():,.0f}, of which '
      f'${gdf["mh_home_exempt_assessed"].sum():,.0f} is exempt under '
      f'C.R.S. 39-3-126.5 on the personal-property roll)')
print(f'effect on modeled current city revenue: '
      f'+${gdf["mh_home_assessed"].sum() * CITY_MILLAGE / 1000:,.0f} '
      f'(the exemption-preserving alternative would add '
      f'${(gdf["mh_home_assessed"].sum() - gdf["mh_home_exempt_assessed"].sum()) * CITY_MILLAGE / 1000:,.0f})')
print('\nParcel-feed MobileHomePresent is unusable as a cross-check — it reads '
      f'"yes" on only {int((county["MobileHomePresent"].astype(str).str.strip().str.lower() == "yes").sum())} '
      'of 101,040 county parcels.')

### Base validation against the assessor's own abstract

The control is Pueblo County's **2025 Tax Authority Abstract by Class**, taxing
authority 4 = City of Pueblo, corroborated by the Colorado Division of Property
Taxation 2025 Annual Report. Two structural facts make the comparison clean:

- **State-assessed property** — public utilities, railroads, telecoms — is valued
  by the state and allocated to districts, never to parcels. Inside the city it is
  **\$376.4M, 24.3% of the roll.** The assessor's own export confirms this
  directly: its 650 `SA` records carry no parcel geometry.
- **Business personal property** is abstracted *inside* the Commercial and
  Industrial lines, so those lines are not comparable to a parcel file. The
  county's is \$214.4M, most of it the EVRAZ Rocky Mountain Steel mill.

**Residential carries no personal property at all** in the Colorado abstract, so
the residential line is a like-for-like test — and it is the one to read.

In [ ]:
# Pueblo County Assessor, 2025 Tax Authority Abstract by Class, authority 4
# (City of Pueblo). Cross-checked against the DPT 2025 Annual Report.
ABSTRACT_CITY_2025 = {
    'Vacant': 31_116_800, 'Residential': 649_173_060, 'Commercial': 351_169_460,
    'Industrial': 137_436_380, 'Agricultural': 1_601_400, 'Natural Resources': 3_110,
    'State Assessed': 376_357_900,
}
ABSTRACT_CITY_TOTAL = 1_546_858_110
CITY_CERTIFIED_GROSS_AV = 1_545_392_060
COUNTY_REAL_PROPERTY = 1_945_172_010      # gross real property, county-wide
COUNTY_PERSONAL_PROPERTY = 214_437_940
COUNTY_STATE_ASSESSED = 526_285_700
assert sum(ABSTRACT_CITY_2025.values()) == ABSTRACT_CITY_TOTAL

model = gdf[gdf['full_exmp'] == 0]
modeled_av = float(model['total_assessed'].sum())
city_non_state = ABSTRACT_CITY_TOTAL - ABSTRACT_CITY_2025['State Assessed']

_cty = county[county['TaxExempt'].astype(str).str.strip().str.lower() != 'yes']
county_av = float((_cty['LandAssessedValue'] + _cty['ImprovementsAssessedValue']).sum())

print('=== WHOLE-FILE CHECK: Pueblo County real property ===')
print(f'  parcel file, all non-exempt county parcels    ${county_av:,.0f}')
print(f'  assessor abstract, county real property       ${COUNTY_REAL_PROPERTY:,.0f}')
print(f'  -> {100 * (county_av / COUNTY_REAL_PROPERTY - 1):+.2f}%')
print(f'  (netted out of the county roll first: state-assessed '
      f'${COUNTY_STATE_ASSESSED:,.0f} + personal property '
      f'${COUNTY_PERSONAL_PROPERTY:,.0f})')

print('\n=== CITY OF PUEBLO ===')
print(f'  modeled taxable assessed value                ${modeled_av:,.0f}')
print(f'  abstract, city total                          ${ABSTRACT_CITY_TOTAL:,.0f}')
print(f'  less state-assessed (never in a parcel file) -${ABSTRACT_CITY_2025["State Assessed"]:,.0f}')
print(f'  = real + business personal property           ${city_non_state:,.0f}')
print(f'  -> {100 * (modeled_av / city_non_state - 1):+.2f}%   residual '
      f'${city_non_state - modeled_av:,.0f} = the city\'s business personal property')
print(f'\n  city certified gross AV (control)             ${CITY_CERTIFIED_GROSS_AV:,.0f}')
print(f'  abstract total vs certified                    '
      f'{100 * (ABSTRACT_CITY_TOTAL / CITY_CERTIFIED_GROSS_AV - 1):+.2f}%  '
      f'(confirms the residential column is on the local-government basis)')

_res_av = float(model.loc[model['stat_class'] == 'residential', 'total_assessed'].sum())
print('\n=== THE LIKE-FOR-LIKE TEST ===')
print(f'  residential, modeled   ${_res_av:,.0f}')
print(f'  residential, abstract  ${ABSTRACT_CITY_2025["Residential"]:,.0f}')
print(f'  -> {100 * (_res_av / ABSTRACT_CITY_2025["Residential"] - 1):+.2f}%  '
      f'(no personal property in this class)')

## Section 4 — Rebuild the current city tax

`current_tax` = the City of Pueblo's own **15.633 mills** applied to the taxable
assessed value of each non-exempt parcel. Nothing else in the ~94-mill stack is
touched.

**The senior/veteran homestead exemption is preserved.** Colorado exempts 50% of
the first \$200,000 of actual value for qualifying seniors and disabled veterans.
The assessor's published `*AssessedValue` fields are **gross** — the exemption is
not netted out of them (senior-flagged parcels show the same 6.25% ratio as
everyone else) — so it is applied here, to both the current and the reformed
tax, so the same households keep the same relief under either system. The state
reimburses local governments for the exemption, so the city's revenue is
unaffected by it; only the distribution across parcels is.

In [ ]:
SENIOR_EXEMPT_ACTUAL = 100_000.0   # 50% of the first $200,000 of actual value

# Dollar exemption is applied to improvements first, then to land (repo convention).
gdf['senior_exempt_actual'] = np.where(
    gdf['senior_exmp'] == 1, np.minimum(gdf['total_actual'], SENIOR_EXEMPT_ACTUAL), 0.0)
_imp_relief = np.minimum(gdf['senior_exempt_actual'], gdf['impr_actual'])
_land_relief = (gdf['senior_exempt_actual'] - _imp_relief).clip(lower=0)

gdf['taxable_impr_actual'] = (gdf['impr_actual'] - _imp_relief).clip(lower=0)
gdf['taxable_land_actual'] = (gdf['land_actual'] - _land_relief).clip(lower=0)

# Re-apply the parcel's own effective assessment rate to the post-exemption actual
# value, so the class rate (6.25% / 27% / a mixed blend) is preserved exactly.
_land_rate = np.where(gdf['land_actual'] > 0,
                      gdf['land_assessed'] / gdf['land_actual'].replace(0, np.nan), 0.0)
_impr_rate = np.where(gdf['impr_actual'] > 0,
                      gdf['impr_assessed'] / gdf['impr_actual'].replace(0, np.nan), 0.0)
_land_rate = np.nan_to_num(_land_rate)
_impr_rate = np.nan_to_num(_impr_rate)

gdf['taxable_land_value'] = (gdf['taxable_land_actual'] * _land_rate).round(2)
gdf['taxable_improvement_value'] = (gdf['taxable_impr_actual'] * _impr_rate).round(2)
gdf['taxable_total_value'] = gdf['taxable_land_value'] + gdf['taxable_improvement_value']
gdf.loc[gdf['full_exmp'] == 1, ['taxable_land_value', 'taxable_improvement_value',
                                'taxable_total_value']] = 0.0

gdf['millage_rate'] = CITY_MILLAGE
current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

_gross_av = float(gdf.loc[gdf['full_exmp'] == 0, 'total_assessed'].sum())
_net_av = float(gdf.loc[gdf['full_exmp'] == 0, 'taxable_total_value'].sum())
print(f'Gross assessed value (non-exempt)        ${_gross_av:,.0f}')
print(f'Senior/veteran exemption                -${_gross_av - _net_av:,.0f} '
      f'({100 * (1 - _net_av / _gross_av):.2f}% of the base)')
print(f'Net taxable assessed value               ${_net_av:,.0f}')
print(f'\nModeled current city revenue at {CITY_MILLAGE} mills: ${current_revenue:,.0f}')

### Revenue gate

The City of Pueblo levies its 15.633 mills against its **whole** taxable base —
real property *plus* business personal property *plus* state-assessed utility,
railroad and telecom property. A parcel file contains only real property, and a
land value tax can only apply to real property, so the quantity this model must
match is the **real-property share of the city's levy**, not the whole levy.

| Figure | Amount | Source |
|---|---|---|
| City gross certified levy, roll 2025 | \$24,159,114 | Colorado DOLA DLG-70 compilation, 15.633 mills × certified AV \$1,545,392,060 |
| City levy on net-of-TIF AV | \$21,680,697 | Ord. 11074, net AV \$1,386,854,524 |
| City budgeted revenue (after \$400k uncollectible allowance) | \$21,280,967 | Ord. 11074 / 2026 Adopted Budget |
| Actual property tax received, 2024 | \$19,176,944 | City of Pueblo 2024 ACFR |
| **State-assessed AV inside the city** | **\$373,000,000** → ≈\$5.83M of levy | DPT 2025 Annual Report — **cannot** be in a parcel file and **cannot** be split-rated |

So the gate that can actually fail is **base validation** (Section 3), not the
revenue arithmetic — at the published rate the revenue figure is an identity.
Base validation lands at roughly −1% on residential, the sharpest single test,
with the remaining gap concentrated in the Commercial and Industrial lines where
DPT folds in business personal property.

In [ ]:
CITY_LEVY_GROSS_2025 = 24_159_114        # DOLA, 15.633 mills x certified AV
CITY_LEVY_NET_TIF_2025 = 21_680_697      # Ord. 11074, net of the URA/TIF increment
STATE_ASSESSED_AV = 373_000_000
state_assessed_levy = STATE_ASSESSED_AV * CITY_MILLAGE / 1000.0

print(f'Modeled real-property city revenue      ${current_revenue:,.0f}')
print(f'+ state-assessed property at {CITY_MILLAGE} mills  ${state_assessed_levy:,.0f}')
print(f'= modeled total city levy                ${current_revenue + state_assessed_levy:,.0f}')
print(f'vs certified gross levy                  ${CITY_LEVY_GROSS_2025:,.0f}  '
      f'({100 * ((current_revenue + state_assessed_levy) / CITY_LEVY_GROSS_2025 - 1):+.2f}%)')
print(f'vs levy on net-of-TIF AV                 ${CITY_LEVY_NET_TIF_2025:,.0f}  '
      f'({100 * ((current_revenue + state_assessed_levy) / CITY_LEVY_NET_TIF_2025 - 1):+.2f}%)')
print()
print(f'Real property is {100 * current_revenue / (current_revenue + state_assessed_levy):.1f}% '
      f'of the modeled city levy — that share is what the split-rate redistributes.')

# Urban-renewal / TIF exposure: the increment on these parcels is diverted to the
# Pueblo Urban Renewal Authority under either tax system.
URA_DISTRICTS = {
    '60BH': 'Eastside URA', '60BM': 'Minnequa Lake URA', '60BT': 'Downtown TIF URA #2',
    '60BW': 'Lower Westside URA', '60BG': 'Union Ave URA', '60BF': 'Union Ave URA',
    '60BV': 'South Santa Fe URA', '60BU': 'North Pueblo URA', '60BS': 'Downtown TIF URA #1',
    '60BD': 'Thunder Village URA', '70VT': 'North Pueblo URA', '70VA': 'Thunder Village URA',
}
gdf['ura'] = gdf['tax_district'].map(URA_DISTRICTS)
_u = gdf[(gdf['ura'].notna()) & (gdf['full_exmp'] == 0)]
print(f'\nUrban-renewal / TIF districts: {len(_u):,} parcels, '
      f'${_u["taxable_total_value"].sum():,.0f} taxable AV '
      f'({100 * _u["taxable_total_value"].sum() / _net_av:.2f}% of the base)')

## Section 5 — 4:1 split-rate model

In [ ]:
# Fully-exempt parcels are held out of the solver AND dropped from the modeled
# output: current = new = $0 carries no signal and would add a meaningless "Exempt"
# bar plus a spike at 0% in the distribution.
n_exempt = int((gdf['full_exmp'] == 1).sum())
gdf = gdf[gdf['full_exmp'] == 0].copy()

for _stale in ['tax_change', 'tax_change_pct', 'new_tax']:
    if _stale in gdf.columns:
        gdf = gdf.drop(columns=[_stale])

land_millage, improvement_millage, new_revenue, gdf = model_split_rate_tax(
    df=gdf,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=float(gdf['current_tax'].sum()),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f'Held out {n_exempt:,} fully-exempt parcels; modelling {len(gdf):,}.')
print(f'\nCurrent uniform city rate : {CITY_MILLAGE:.4f} mills')
print(f'Land millage              : {land_millage:.4f} mills  '
      f'({land_millage / CITY_MILLAGE:.2f}x current)')
print(f'Improvement millage       : {improvement_millage:.4f} mills  '
      f'({improvement_millage / CITY_MILLAGE:.2f}x current)')
print(f'Ratio check               : {land_millage / improvement_millage:.4f}:1')
print(f'\nRevenue: ${new_revenue:,.0f} new vs ${current_revenue:,.0f} current '
      f'({100 * (new_revenue / current_revenue - 1):+.4f}%)')

_L = gdf['taxable_land_value'].sum()
_I = gdf['taxable_improvement_value'].sum()
print(f'\nTaxable base: land ${_L:,.0f} ({100 * _L / (_L + _I):.1f}%)  '
      f'improvements ${_I:,.0f} ({100 * _I / (_L + _I):.1f}%)')

In [ ]:
category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary,
                           title=f'{CITY_NAME} — 4:1 split-rate on the city levy')

### Gate 5 — artifact scan

Read the table above like a sceptic before trusting it. The canaries are
distinct categories piling up on one extreme value, built-up categories showing
~0% building share, or a direction that contradicts economics.

In [ ]:
d = gdf.copy()
d['bldg_share'] = (d['taxable_improvement_value']
                   / (d['taxable_land_value'] + d['taxable_improvement_value'])
                   .replace(0, np.nan))
scan = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_zero_bldg=('taxable_improvement_value', lambda s: (s <= 1000).mean()),
    median_bldg_sqft=('bldg_sqft', 'median'),
    pct_with_footprint=('bldg_count', lambda s: (s > 0).mean()),
).sort_values('median_pct', ascending=False)
print(scan.round(3).to_string())

# Placeholder-improvement check: a recorded structure carrying a token value is the
# artifact that wrecked the Denver run. Pueblo is checked for it explicitly.
_built = d[d['bldg_count'] > 0]
print(f'\nParcels with a building footprint but <=$1,000 improvement value: '
      f'{int((_built["impr_actual"] <= 1000).sum()):,} of {len(_built):,} '
      f'({100 * (_built["impr_actual"] <= 1000).mean():.2f}%)')
print(f'Parcels with a footprint but $0 improvement value: '
      f'{int((_built["impr_actual"] <= 0).sum()):,}')
print(f'Parcels with NO footprint but a positive improvement value: '
      f'{int(((d["bldg_count"] == 0) & (d["impr_actual"] > 0)).sum()):,}')

# Land-share artifact check: a flat assessor land ratio (the Phoenix failure) would
# show up as a spike in the land-share distribution.
_imp = d[d['impr_actual'] > 0].copy()
_imp['land_share_actual'] = _imp['land_actual'] / _imp['total_actual']
print(f'\nLand share of actual value, improved parcels — '
      f'p10 {_imp["land_share_actual"].quantile(.10):.3f}  '
      f'median {_imp["land_share_actual"].median():.3f}  '
      f'p90 {_imp["land_share_actual"].quantile(.90):.3f}')
_top = _imp['land_share_actual'].round(3).value_counts().head(3)
print(f'Most common rounded land shares (a flat ratio would spike here): '
      f'{dict(_top)}')

## Section 6 — Deck figures

Everything below feeds the Landslide manifest.

In [ ]:
n = len(gdf)
cut = int((gdf['tax_change'] < -0.005).sum())
up = int((gdf['tax_change'] > 0.005).sum())
same = n - cut - up
print(f'Parcels modeled      {n:,}')
print(f'  pay less           {cut:,} ({100 * cut / n:.1f}%)')
print(f'  pay more           {up:,} ({100 * up / n:.1f}%)')
print(f'  unchanged          {same:,}')
print(f'Median change        {gdf["tax_change_pct"].median():+.1f}%  '
      f'(${gdf["tax_change"].median():+,.0f})')
print(f'Share up >10%        {100 * (gdf["tax_change_pct"] > 10).mean():.1f}%')
print(f'Share down >10%      {100 * (gdf["tax_change_pct"] < -10).mean():.1f}%')

# The median detached single-family home, with the spread. Never quote the median
# alone: a near-neutral median can hide a very wide distribution.
sf = gdf[gdf['PROPERTY_CATEGORY'] == 'Single Family Residential']
print(f'\n--- Single Family Residential (detached), n={len(sf):,} ---')
print(f'median change            {sf["tax_change_pct"].median():+.2f}%  '
      f'(${sf["tax_change"].median():+,.0f})')
print(f'median current city tax  ${sf["current_tax"].median():,.0f}')
print(f'median new city tax      ${sf["new_tax"].median():,.0f}')
print(f'median actual value      ${sf["total_actual"].median():,.0f}')
print(f'median land value        ${sf["land_actual"].median():,.0f}')
print(f'median lot              {sf["lot_sqft"].median():,.0f} sqft')
print(f'share paying less        {100 * (sf["tax_change"] < 0).mean():.1f}%')
print(f'share up >10%            {100 * (sf["tax_change_pct"] > 10).mean():.1f}%')
print(f'share down >10%          {100 * (sf["tax_change_pct"] < -10).mean():.1f}%')

print('\n--- by category (median %, n) ---')
_cat = gdf.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_dollars=('tax_change', 'median'),
    net_change=('tax_change', 'sum'),
).sort_values('median_pct', ascending=False)
print(_cat.round(1).to_string())

# Underdeveloped land tiers — the core LVT story.
print('\n--- underdeveloped land ---')
tot_land_actual = gdf['land_actual'].sum()
_imp_ratio = gdf['impr_actual'] / gdf['total_actual'].replace(0, np.nan)
for label, mask in [
    ('vacant (zero improvement)', gdf['impr_actual'] <= 0),
    ('improvement < 10% of value', _imp_ratio < 0.10),
    ('improvement < 25% of value', _imp_ratio < 0.25),
    ('improvement < 50% of value', _imp_ratio < 0.50),
]:
    s = gdf[mask.fillna(False)]
    if not len(s):
        continue
    print(f'  {label:30s} n={len(s):6,}  land ${s["land_actual"].sum() / 1e6:8.1f}M  '
          f'{100 * s["land_actual"].sum() / tot_land_actual:5.1f}% of city land value  '
          f'median {s["tax_change_pct"].median():+7.1f}%  '
          f'net ${s["tax_change"].sum():+,.0f}')

### Underdeveloped land — measured by building coverage, not improvement ratio

Every other city in this repo tiers "underdeveloped" land by **improvement ratio**
(improvement value as a share of total value). That measure does not work in
Pueblo. Land is only **16.0%** of the taxable base and the median improved parcel
carries land at just **9.3%** of its value, so almost nothing clears an
"improvement < 50% of value" test — it captures 15% of the city's land value
where the same test in Denver captures 49%. The measure is saturated, not
informative.

So Pueblo is tiered on **building coverage** — the physical measure — defined as

> `building coverage = building area (sqft) / lot area (sqft)`

using the assessor's own `Primary Building Area` where present and the Overture
footprint sum where it is not. Both tierings are reported below; the
building-coverage one is the one the deck should use, and any slide resting on it
must say it is a coverage measure.

In [ ]:
# Building area: assessor's own figure first, Overture footprint as fallback.
gdf['building_area_sqft'] = np.where(gdf['bldg_area_sqft'] > 0,
                                     gdf['bldg_area_sqft'], gdf['bldg_sqft'])
gdf['building_coverage'] = (gdf['building_area_sqft']
                            / gdf['lot_sqft'].replace(0, np.nan)).fillna(0.0)
gdf['area_source'] = np.where(gdf['bldg_area_sqft'] > 0, 'assessor',
                              np.where(gdf['bldg_sqft'] > 0, 'overture', 'none'))
print('building-area source:')
print(gdf['area_source'].value_counts().to_string())
print(f'\nbuilding coverage — median {gdf["building_coverage"].median():.3f}, '
      f'p90 {gdf["building_coverage"].quantile(.9):.3f}')

tot_land = gdf['land_actual'].sum()
print('\n--- A. improvement-ratio tiers (the repo standard — SATURATED here) ---')
_ir = gdf['impr_actual'] / gdf['total_actual'].replace(0, np.nan)
for label, mask in [('vacant (zero improvement)', gdf['impr_actual'] <= 0),
                    ('improvement < 10% of value', _ir < 0.10),
                    ('improvement < 25% of value', _ir < 0.25),
                    ('improvement < 50% of value', _ir < 0.50)]:
    s = gdf[mask.fillna(False)]
    print(f'  {label:30s} n={len(s):6,}  land ${s["land_actual"].sum() / 1e6:8.1f}M  '
          f'{100 * s["land_actual"].sum() / tot_land:5.1f}% of city land value  '
          f'median {s["tax_change_pct"].median():+7.1f}%  '
          f'net ${s["tax_change"].sum():+,.0f}')

print('\n--- B. building-coverage tiers (USE THESE for Pueblo) ---')
UNDERDEV_TIERS = [
    ('no building at all', gdf['building_area_sqft'] <= 0),
    ('coverage < 5%', gdf['building_coverage'] < 0.05),
    ('coverage < 10%', gdf['building_coverage'] < 0.10),
    ('coverage < 20%', gdf['building_coverage'] < 0.20),
]
underdev_stats = {}
for label, mask in UNDERDEV_TIERS:
    s = gdf[mask]
    underdev_stats[label] = dict(
        parcels=int(len(s)), land_value=float(s['land_actual'].sum()),
        pct_of_city_land=round(100 * s['land_actual'].sum() / tot_land, 1),
        median_change_pct=round(float(s['tax_change_pct'].median()), 1),
        net_change=float(s['tax_change'].sum()))
    print(f'  {label:22s} n={len(s):6,}  land ${s["land_actual"].sum() / 1e6:8.1f}M  '
          f'{100 * s["land_actual"].sum() / tot_land:5.1f}% of city land value  '
          f'median {s["tax_change_pct"].median():+7.1f}%  '
          f'net ${s["tax_change"].sum():+,.0f}')

### The scope caveat — mandatory on every slide

The City of Pueblo's own levy is **15.633 of the ~94.077 mills** on a typical
city parcel — **16.6%**. Every percentage in this notebook is a change in *that
16.6% slice*, not in the whole property-tax bill. County, School District 60,
the Library District, the Water Conservancy district and the Conservancy District
are all held unchanged.

**Slide-ready framing:**

> *Pueblo's own levy is 15.633 of the ~94 mills on a city property — about one
> sixth of the bill. Re-striking just that sixth at 4:1, the median Pueblo home's
> city portion falls 12.7% — roughly \$28 a year — while the whole-bill change is
> about one sixth of that.*

Never write "your property taxes fall 12.7%" unqualified.

In [ ]:
TOTAL_STACK_MILLS = 94.077     # typical City of Pueblo parcel, tax district 60B
city_share = CITY_MILLAGE / TOTAL_STACK_MILLS
sf = gdf[gdf['PROPERTY_CATEGORY'] == 'Single Family Residential']
_sf_med_pct = float(sf['tax_change_pct'].median())
_sf_med_dol = float(sf['tax_change'].median())

print(f'City levy {CITY_MILLAGE} of ~{TOTAL_STACK_MILLS} mills = '
      f'{100 * city_share:.1f}% of the stack')
print(f'Median single-family home: city portion {_sf_med_pct:+.1f}% '
      f'(${_sf_med_dol:+,.0f}/yr)')
print(f'  -> as a share of the WHOLE bill that is about '
      f'{_sf_med_pct * city_share:+.1f}%')
print(f'  median whole-bill estimate: '
      f'${sf["total_assessed"].median() * TOTAL_STACK_MILLS / 1000:,.0f}/yr, '
      f'of which the city portion is ${sf["current_tax"].median():,.0f}')

In [ ]:
import matplotlib
matplotlib.use('Agg')

fig, ax = plt.subplots(figsize=(10, 6))
_s = gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values()
_s.plot.barh(ax=ax, color=np.where(_s < 0, '#2E7D32', '#8B0000'))
ax.axvline(0, color='black', linewidth=1)
ax.set_title(f'Pueblo — median city-tax change by category (4:1 split-rate)')
ax.set_xlabel('Median % change in the City of Pueblo portion of the bill')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print('wrote', DATA_DIR / 'category_preview.png')

## Section 7 — Census join + export

In [ ]:
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

# Parcel-map export
from lvt.parcel_map import save_parcel_map_export, create_parcel_map
map_gdf = save_parcel_map_export(
    gdf=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
)
create_parcel_map(map_gdf, CITY_NAME)
print('Done.')

### `data/parcels.gpq` — the modeled per-parcel file

Written for downstream consumers (the Landslide example-scout stage). Column
names follow that tool's canonical schema so it needs no `--colmap`: `PARCELID`,
`CNTLNDVAL`, `CNTIMPVAL`, `USEDSCRP`, `ZONE`, `SITEADDRESS`, `LIVUNIT`, plus
`lot_sqft` and `bldg_sqft` so tax per square foot can be computed on either
land or building area.

In [ ]:
export_cols = {
    # canonical names the Landslide example-scout expects (no --colmap needed)
    'PARCELID': gdf['PAR_TXT'].astype(str),
    'CNTLNDVAL': gdf['taxable_land_value'],
    'CNTIMPVAL': gdf['taxable_improvement_value'],
    'USEDSCRP': gdf['PROPERTY_CATEGORY'].astype(str),
    'CLASSCD': gdf['abstract_code'].astype(str),
    'ZONE': gdf['ZONING'].astype(str),
    'SITEADDRESS': gdf['SITEADDRESS'].astype(str),
    'LIVUNIT': gdf['est_units'],
    'OWNER_NAME': gdf['Owner'].astype(str),
    # values
    'land_actual': gdf['land_actual'],
    'improvement_actual': gdf['impr_actual'],
    'total_actual': gdf['total_actual'],
    'land_assessed': gdf['land_assessed'],
    'improvement_assessed': gdf['impr_assessed'],
    # areas
    'lot_sqft': gdf['lot_sqft'],
    'lot_acres': gdf['lot_acres'],
    'building_area_sqft': gdf['building_area_sqft'],
    'building_coverage': gdf['building_coverage'],
    'area_source': gdf['area_source'],
    'year_built': gdf['year_built'],
    'bldg_count': gdf['bldg_count'],
    # classification provenance
    'abstract_code': gdf['abstract_code'].astype(str),
    'account_type': gdf['account_type'].astype(str),
    'nbhd_group': gdf['nbhd_group'].astype(str),
    'stat_class': gdf['stat_class'].astype(str),
    'PROPERTY_CATEGORY': gdf['PROPERTY_CATEGORY'].astype(str),
    'is_parking_lot': gdf['is_parking_lot'],
    'parking_coverage': gdf['parking_coverage'],
    'is_mobile_home_park': gdf['is_mobile_home_park'],
    # manufactured homes attributed from the assessor's own MH accounts
    'mh_home_count': gdf['mh_home_count'],
    'mh_home_actual': gdf['mh_home_actual'],
    'mh_home_assessed': gdf['mh_home_assessed'],
    'mh_home_exempt_assessed': gdf['mh_home_exempt_assessed'],
    'addr_count': gdf['addr_count'],
    'unit_count': gdf['unit_count'],
    'neighborhood': gdf['Neighborhood'].astype(str),
    'tax_district': gdf['tax_district'].astype(str),
    'ura_district': gdf['ura'].fillna('').astype(str),
    'senior_exmp': gdf['senior_exmp'],
    'full_exmp': gdf['full_exmp'],
    'exemption_codes': gdf['exemption_codes'].fillna('').astype(str),
    # tax outcome
    'taxable_land_value': gdf['taxable_land_value'],
    'taxable_improvement_value': gdf['taxable_improvement_value'],
    'millage_rate': gdf['millage_rate'],
    'current_tax': gdf['current_tax'],
    'new_tax': gdf['new_tax'],
    'tax_change': gdf['tax_change'],
    'tax_change_pct': gdf['tax_change_pct'],
}
for _c in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
    export_cols[_c] = gdf[_c] if _c in gdf.columns else np.nan

parcels_out = gpd.GeoDataFrame(pd.DataFrame(export_cols),
                               geometry=gdf.geometry.values, crs=gdf.crs)
parcels_out['tax_per_lot_sqft'] = (parcels_out['current_tax']
                                   / parcels_out['lot_sqft'].replace(0, np.nan))
parcels_out['tax_per_bldg_sqft'] = (parcels_out['current_tax']
                                    / parcels_out['building_area_sqft'].replace(0, np.nan))
parcels_out.to_parquet(DATA_DIR / 'parcels.gpq', index=False)
print(f'wrote {DATA_DIR / "parcels.gpq"}  {parcels_out.shape}')
print(f'  geometry valid        {100 * parcels_out.geometry.is_valid.mean():.1f}%')
print(f'  lot_sqft > 0          {100 * (parcels_out["lot_sqft"] > 0).mean():.1f}%')
print(f'  building_area_sqft >0 {100 * (parcels_out["building_area_sqft"] > 0).mean():.1f}%')
print(f'  SITEADDRESS populated {100 * (parcels_out["SITEADDRESS"].str.len() > 0).mean():.1f}%')
print(f'  categories: {parcels_out["USEDSCRP"].nunique()}')

## Validation summary

| Check | Result |
|---|---|
| Levy scope | City of Pueblo's own 15.633 mills only (16.6% of the ~94.077-mill stack) |
| Roll year | 2025 (payable 2026), derived from the assessment rates in the file and confirmed against DPT |
| Assessment formula | Not assumed — reverse-engineered: residential 6.25%, non-residential 27.00%, flat, no actual-value exclusion; mixed-use parcels carry a blend |
| Revenue | Modeled real-property levy at the **published** rate; revenue-neutral by construction |
| Base validation | See Section 3 — modeled residential \$649,001,831 vs the assessor's 2025 abstract \$649,173,060, **−0.03%**; the Commercial/Industrial gap is business personal property, which no parcel file contains |
| Manufactured homes | Section 3d — all 1,293 city `MH` accounts joined to the parcel they stand on (100%), \$33.36M of home value / \$2.08M of assessed value put back on 118 parcels |
| Classification | Zoning (100% matched) + the assessor's statutory class + address-point facility class and unit counts |
| Placeholder improvement values | Checked — effectively absent in Pueblo (unlike Denver) |
| Flat-land-ratio artifact | Checked — land share is a continuous distribution, no spike |
| Census coverage | see Section 7 |

### Limitations

1. **No assessor use-code.** Pueblo County's open feed publishes no property
   class, land-use or abstract code, so `PROPERTY_CATEGORY` is built from
   zoning + statutory assessment class + addressing data. The
   residential/non-residential split is the assessor's own and is solid; the
   finer splits (single-family vs. 2-4 unit vs. 5+ unit, retail vs. office) rest
   on zoning and dwelling-unit counts and will misfile individual parcels.
2. **Business personal property and state-assessed property are outside the
   model.** They pay the city's mills but cannot be land-value-taxed and are not
   in any parcel file. Real property is ~74% of the city's levy.
3. **TIF.** The city's certified levy is struck on assessed value net of the
   urban-renewal increment diverted to the Pueblo Urban Renewal Authority. The
   URA districts identifiable from tax-district codes hold ~9% of the modeled
   base, but the certified 2025 increment (\$159.98M) exceeds the total assessed
   value of those coded parcels, so some increment sits in parcels not
   distinguishable by district code. Diversion applies under either tax system,
   so it does not bias the percentage changes.
4. **Percentage changes are roll-year invariant; dollar magnitudes are on the
   2025 roll.**
5. **Manufactured homes are modeled as improvements, gross of the \$28,000
   low-value exemption** (C.R.S. 39-3-126.5), on both sides of the comparison.
   The exemption covers 906 of the 1,293 city homes and \$684,610 of assessed
   value; carrying it across instead would leave modeled current revenue
   \$10,702 lower and put the 94 individual manufactured-housing lots back at
   the land-only ceiling. The choice is stated on the slide, not buried.